## 1.加载dataset

In [1]:
from datasets import load_dataset

base_dir = r".\RubricHub_v1"

# streaming=True
ds = load_dataset(
    "parquet",
    data_files={"train": base_dir + r"\**\*.parquet"},
    streaming=True
)

print(ds)
print(next(iter(ds["train"])))

IterableDatasetDict({
    train: IterableDataset({
        features: ['prompt', 'data_source', 'ability', 'reward_model', 'extra_info', 'Rubrics', '__index_level_0__'],
        num_shards: 7
    })
})
{'prompt': [{'content': "Please read the following text and summarize it. The final output should be limited to 50 words. Here is the text: NAME_1 has been granted extra time to decide whether to compete in the World Cross-Country Championships.The 31-year-old is concerned the event, which starts on 19 March in France, could upset her preparations for the London Marathon on 17 April. 'There is no question that NAME_2 would be a huge asset to the GB team,' said NAME_3 of UK Athletics. 'But she is working out whether she can accommodate the worlds without too much compromise in her marathon training.' NAME_4 must make a decision by Tuesday - the deadline for team nominations. British team member NAME_5 said the team would understand if NAME_4 opted out of the event. 'It would be fantastic t

## 2.使用LLM对Prompt（Query）生成Answer
+ LLM选项：GPT-4o、deepseek-r1、qwen_plus

In [2]:
# API
import openai
import time
import json
class Get:
    def __init__(self):
        self.prompt = ""
    def calc(self, query, temp=1, n=1, model='3.5', pic_urls=None):
        print(model)
        # model候选 = ['o1','qwen_vl','qwen_plus','claude','3.5','4o','4omini']
        openai.api_type = "azure"  # 'azure', 'azure_ad', 'open_ai'
        openai.api_base = "https://runway.devops.xiaohongshu.com"
        openai.api_version = "2023-05-15"  # "2023-03-15-preview"
        if model == 'gemini':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/gemini/v1/chat/completions'
            payload = json.dumps({
                "temperature": temp,
                "model":"gemini-2.0-flash-thinking-exp-01-21",
                'n':n,
                # "model": "deepseek-r1",
                "messages": [
                        {
                            "role": "user",
                            "content": query
                        }
                    ]
            })
            headers = {
                    'Content-Type': 'application/json',
                    'api-key': 'd9c75e199caf4ec0b248b06ac2edd668'
                }
            count = 0
            while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = [tem['message']['content'] for tem in kk['choices']]
                        return re_to_store, {'prompt': 0, 'completion': 0}
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            return [], {'prompt': 0, 'completion': 0}
                        count += 1
                        time.sleep(4)

        if model == 'deepseek-r1-qwen':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/qwen/v1/chat/completions'
            payload = json.dumps({
                "temperature": temp,
                "model":"deepseek-r1-distill-qwen-32b",
                # "model": "deepseek-r1",
                "messages": [
                        {
                            "role": "system",
                            "content": "你是一个帮助用户查找信息的 AI 助手。"
                        },
                        {
                            "role": "user",
                            "content": query
                        }
                    ]
            })
            headers = {
                    'Content-Type': 'application/json',
                    'api-key': '13920d86c8234d5b992a9e4962e96eaf'
                }
            to_res = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['choices'][0]['message']['content']
                        to_res.append(re_to_store)
                        n-=1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_res.append("")
                            n -= 1
                            break
                        count += 1
                        time.sleep(4)
            return to_res, {'prompt': 0, 'completion': 0}

        if model == 'deepseek-r1':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/qwen/v1/chat/completions'
            payload = json.dumps({
                "temperature": temp,
                "model":"deepseek-r1",
                "messages": [
                        {
                            "role": "system",
                            "content": "你是一个帮助用户查找信息的 AI 助手。"
                        },
                        {
                            "role": "user",
                            "content": query
                        }
                    ]
            })

            headers = {
                    'Content-Type': 'application/json',
                    'api-key': 'f408303a863e4871ba40ee135808c0ee'
                }
            to_res = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['choices'][0]['message']['content']
                        to_res.append(re_to_store)
                        n-=1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_res.append("")
                            n -= 1
                            break
                        count += 1
                        time.sleep(4)
            return to_res, {'prompt': 0, 'completion': 0}
        if model == 'o1':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/chat/completions?api-version=2024-12-01-preview'
            payload = json.dumps({
                "temperature": temp,
                "messages": [
                    {
                        "role": "user",
                        "content": query
                    }
                ]
            })
            headers = {
                # 'Authorization': '',
                'Content-Type': 'application/json',
                'api-key': '81080d19041443ca84e678fea3aea125'
            }
            to_return = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['choices'][0]['message']['content']
                        to_return.append(re_to_store)
                        n = n-1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_return.append("")
                            n = n-1
                            break
                        count += 1
                        time.sleep(4)
            return to_return, {'prompt': 0, 'completion': 0}
        if model == 'o1-mini':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/chat/completions?api-version=2024-10-01-preview'
            payload = json.dumps({
                "temperature": temp,
                "messages": [
                    {
                        "role": "user",
                        "content": query
                    }
                ]
            })
            headers = {
                # 'Authorization': '',
                'Content-Type': 'application/json',
                'api-key': '46d94ad4e0a54b9781ad6bad53c9d8e0'
            }
            to_return = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['choices'][0]['message']['content']
                        to_return.append(re_to_store)
                        n = n-1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_return.append("")
                            n = n-1
                            break
                        count += 1
                        time.sleep(4)
            return to_return, {'prompt': 0, 'completion': 0}
        if model == 'o3-mini':
            import requests
            url = 'https://runway.devops.xiaohongshu.com/openai/chat/completions?api-version=2024-12-01-preview'
            payload = json.dumps({
                # "temperature": temp,
                "messages": [
                    {
                        "role": "user",
                        "content": query
                    }
                ]
            })
            headers = {
                # 'Authorization': '',
                'Content-Type': 'application/json',
                'api-key': '86df655bad8a4d90b1c3d821af82eed8'
            }
            to_return = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['choices'][0]['message']['content']
                        to_return.append(re_to_store)
                        n = n-1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_return.append("")
                            n = n-1
                            break
                        count += 1
                        time.sleep(4)
            return to_return, {'prompt': 0, 'completion': 0}
        if model == 'qwen_vl':
            import requests
            url = "https://runway.devops.xiaohongshu.com/openai/qwen/v1/services"
            to_input = {
                "role": "user",
                "content": [
                    {'text': query}
                ]
            }
            if pic_urls is not None:
                for the_url in pic_urls:
                    to_input["content"].append({'image': the_url})
            payload = json.dumps({
                "model": "qwen-vl-plus-0125",
                "input": {
                    "temperature": temp,
                    "top_p": 0.9,
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are a helpful assistant."
                        },
                        to_input
                    ]
                }
            })
            headers = {
                'Authorization': '',
                'Content-Type': 'application/json',
                'api-key': '86568f0942ec41eb800e084e27c3251a'
            }
            count = 0
            while True:
                try:
                    response = requests.request("POST", url, headers=headers, data=payload)
                    kk = json.loads(response.text)
                    re_to_store = kk['output']['choices'][0]['message']['content'][0]['text']
                    return [re_to_store], {'prompt': 0, 'completion': 0}
                except:
                    print("Sleep 4s:{}".format(response.reason))
                    if count > 15:
                        return [""], {'prompt': 0, 'completion': 0}
                    count += 1
                    time.sleep(4)
        if model == 'claude':
            import requests
            url = 'https://runway.devops.rednote.life/openai/bedrock_runtime/model/invoke'
            payload = json.dumps({
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": 2048,
                "temperature": temp,
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": query
                            }
                        ]
                    }
                ]
            })
            headers = {
                # 'Authorization': '',
                'Content-Type': 'application/json',
                'token': '7527c451a9484a639c7f65c3600bb33f'
            }
            to_res = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['content'][0]['text']
                        to_res.append(re_to_store)
                        n-=1
                        break

                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_res.append("")
                            n-=1
                            break
                        count += 1
                        time.sleep(4)
            return to_res, {'prompt': 0, 'completion': 0}
        if model == 'qwen_plus':
            import requests
            url = "https://runway.devops.xiaohongshu.com/openai/qwen/v1/services"
            payload = json.dumps({
                "model": "qwen-plus",
                "input": {
                    "temperature": temp,
                    "top_p": 0.9,
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are a helpful assistant."
                        },
                        {
                            "role": "user",
                            "content": query
                        }
                    ]
                }
            })
            headers = {
                'Authorization': '',
                'Content-Type': 'application/json',
                'api-key': 'a9d346f9d1dc4a56ba061f6bfb96479c'
            }
            to_res = []
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        re_to_store = kk['output']['text']
                        to_res.append(re_to_store)
                        n-=1
                        break
                    except:
                        print("Sleep 4s:{}".format(response.reason))
                        if count > 15:
                            to_res.append("")
                            n-=1
                            break
                        count += 1
                        time.sleep(4)
            return to_res, {'prompt': 0, 'completion': 0}
        if model == 'qwen3.5-plus':
            import requests
            url = 'https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions'
            
            # 请在下面填入你的百炼 API_KEY，或者通过环境变量获取
            api_key = "sk-7d3fa8a33cf44ee091862422c2082002"
            
            payload = json.dumps({
                "model": "qwen3.5-plus",
                "temperature": temp,
                "messages":[
                    {
                        "role": "user",
                        "content": query
                    }
                ],
                "enable_thinking": False  # 对应原代码里的 extra_body={"enable_thinking": True}
            })
            
            headers = {
                'Authorization': f'Bearer {api_key}',
                'Content-Type': 'application/json'
            }
            
            to_res =[]
            while n:
                count = 0
                while True:
                    try:
                        response = requests.request("POST", url, headers=headers, data=payload)
                        kk = json.loads(response.text)
                        
                        # 非流式直接解析 message 内容
                        message = kk['choices'][0]['message']
                        content = message.get('content', '')
                        reasoning = message.get('reasoning_content', '')
                        
                        # 拼装思考过程和最终回复
                        if reasoning:
                            re_to_store = f"====================思考过程====================\n{reasoning}\n====================完整回复====================\n{content}"
                        else:
                            re_to_store = content
                            
                        to_res.append(re_to_store)
                        n -= 1
                        break
                    except Exception as e:
                        print("Sleep 4s:{}".format(e))
                        if count > 15:
                            to_res.append("")
                            n -= 1
                            break
                        count += 1
                        time.sleep(4)
            return to_res, {'prompt': 0, 'completion': 0}
        # ▲▲▲▲▲ 新代码结束 ▲▲▲▲▲
        else:
            if model == '3.5':
                openai.api_key = "f408303a863e4871ba40ee135808c0ee"
                id = "gpt-35-turbo"
            elif model == '4':
                openai.api_key = "c6a8dffac7da4785a68a527dd42292d6"
                id = "gpt4-PTU"
            elif model == '4.5':
                openai.api_key = "85d5e6d1dcc143e9abf2a340622cf480"
                id = "gpt-4"
            elif model == '3.5_16k':
                openai.api_key = "f501ac6ff64742598d6c8b393c2f21f3"
                id = "gpt-35-turbo-16k"
            elif model == '4o':
                openai.api_key = 'd43f29d8ab714d999c26ad8ab2e33556'
                id = "gpt-4o"
            elif model == '4omini':
                openai.api_key = '501018c16a154147b76f03e7e9563565'
                id = "gpt-4o-mini"
            count = 0
            if pic_urls is None:
                cur_message = [{"role": "user", "content": query}]
            else:
                cur_message = [
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "text",
                                        "text": query
                                    }
                                ]
                            }
                        ]
                for cur_url in pic_urls:
                    tem = {
                                        "type": "image_url",
                                        "image_url": {
                                            "url": cur_url
                                        }
                                    }
                    cur_message[0]['content'].append(tem)
            while True:
                try:
                    response = openai.ChatCompletion.create(
                        deployment_id=id,
                        messages=cur_message,
                        temperature=temp,
                        top_p=0.9,
                        n=n
                    )

                    if 'error' in response.keys() and response['error']['code'] == 'context_length_exceeded':
                        print('context_length_exceeded')
                        return [""], {'prompt': 0, 'completion': 0}
                    if 'error' in response.keys() and response['error']['code'] == 'invalid_prompt':
                        print('invalid_prompt')
                        return [""], {'prompt': 0, 'completion': 0}
                    if 'content_filter' == response["choices"][0]["finish_reason"]:
                        print('content_filter')
                        return [""], {'prompt': 0, 'completion': 0}
                    if 'error' in response.keys(): print(response['error']['code'])
                    res = [tem["message"]["content"] for tem in response["choices"]]
                    return res, {'prompt': response['usage']['prompt_tokens'],
                                 'completion': response['usage']['completion_tokens']}
                except Exception as e:
                    if count > 15:
                        return [""], {'prompt': 0, 'completion': 0}
                    print("An error occurred:", e)
                    count += 1
                    time.sleep(4)
# import time
# def main():
#     model = Get()
#     for mod in ['qwen_vl','deepseek-r1','deepseek-r1-qwen','claude','qwen_plus','o3-mini','o1','4omini','4o','qwen_vl']:
#         begin = time.time()
#         tem,_ = model.calc("中国的首都是哪里",model=mod,n=2)
#         cost = time.time()-begin
#         print(mod)
#         print(tem[0])
#         print(cost)

#     # 文本+图片输入示例
#     # for mod in ['qwen_vl','4o','4omini']:
#     #     tem,_ = model.calc('一共有几个图片？它们图片一样吗？分别是什么画面？',model=mod,pic_urls=['https://dashscope.oss-cn-beijing.aliyuncs.com/images/dog_and_girl.jpeg','https://dashscope.oss-cn-beijing.aliyuncs.com/images/dog_and_girl.jpeg'])
#     #     print(tem[0])

# if __name__ == "__main__":
#     main()



In [ ]:
# 选100条数据，设置gen-model和judge-model

#  选前100个Query生成Answer
from itertools import islice
from datasets import Dataset

first_100_list = list(islice(ds["train"], 100))
first_100 = Dataset.from_list(first_100_list)

gen_models = ['4o','deepseek-r1','qwen_plus']
# gen_models = ['qwen3.5-plus'] # for test

judge_models = ['4o','o1','o3mini','deepseek-r1','qwen_plus']
# judge_models = ['qwen3.5-plus'] # for test

model = Get()

In [ ]:
# 如果存在文件“model_res.json”就从文件中直接加载responses
# 否则就生成responses并保存到“model_res.json”中
import os
responses = []

if os.path.exists("model_res.json"):
    print("成功加载")
    responses = json.load(open("model_res.json", "r", encoding='utf-8'))
else:
    print("正在生成...")
    for i, item in enumerate(first_100):
        responses.append({})
        
        for mod in gen_models:
            tem,_ = model.calc(item['prompt'][0]['content'],model=mod,n=1)
            responses[i][mod] = tem[0]
        # break # for testing, 先生成一条就保存，后续再生成剩下的

    # 保存结果
    json.dump(responses, open("model_res.json", "w", encoding='utf-8'), ensure_ascii=False, indent=4)

正在生成...
qwen3.5-plus
Sleep 4s:HTTPSConnectionPool(host='dashscope.aliyuncs.com', port=443): Max retries exceeded with url: /compatible-mode/v1/chat/completions (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))
Sleep 4s:HTTPSConnectionPool(host='dashscope.aliyuncs.com', port=443): Max retries exceeded with url: /compatible-mode/v1/chat/completions (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1017)')))


In [5]:
print(responses)

[{'qwen3.5-plus': 'NAME_1 has extra time until Tuesday to decide on competing in the World Cross-Country Championships, fearing it may disrupt her London Marathon training. While UK Athletics and teammates value her potential contribution, they fully understand her dilemma and support her final choice.'}]


## 3.评分
+ Judge选用：GPT-4o,GPT-o1,o3-mini,deepseek-r1,qwen_plus

### 3.1 重复采样对鲁棒性的提升
+  采样次数$n=16$
+  `result[i][generation_model][judge_model][j][k]`:第 $i$ 条 $data$ ，用 $generation\_model$ 生成 $response$ ，用 $judge\_model$ 根据 $Rubrics[j]$ 进行 $judge$ ，第$k次Judge$ 的结果。

In [ ]:
# 加载评分模板
js = json.load(open("prompt.json", "r", encoding='utf-8'))
print(js['list-grader-template'])

# 对整个rubric-list的一次judge,采样n次，返回list
def get_score(query,response,rubric_list,judge_model,n):
    
    template  = js['list-grader-template']

    prompt = template.replace("{{QUERY}}", query) \
                           .replace("{{RESPONSE}}", response) \
                           .replace("{{RUBRIC}}", rubric_list)
    # Judge n 次
    res, _ = model.calc(prompt, model=judge_model, n=n)

    # 把json格式的字符串res[i]解析成字典ret[i]
    ret = []
    for item in res:
        ret.append(json.loads(item)['scoring_details'])

    # 返回一个列表ret
    # ret[i][j]表示第i次评分中rubric_list的第j项的得分
    return ret


# Role
你是一名严谨的大模型质量评估专家（Judge）。你的核心任务是根据给定的 **Rubric 考点清单**，对 AI 助手的回答进行 **二元判定（Binary Evaluation）**。

# Scoring Philosophy (0/Weight 机制)
本评估采用极度严格的非黑即白评分逻辑：
1. **正向得分项 (Weight > 0)**：
   - **满足**：回答完全涵盖了考点要求的所有细节和条件。得分 = **Weight**。
   - **不满足**：回答缺失、错误或仅部分满足（即使完成了 90% 也视为不满足）。得分 = **0**。
2. **负向扣分项 (Weight < 0)**：
   - **触发**：回答中出现了考点所禁止的内容或错误。得分 = **Weight**（即扣除对应分数）。
   - **未触发**：回答未出现相关负面信息。得分 = **0**。

# Evaluation Workflow
1. **解析 Rubric**：阅读每一个考点，明确其“通过”的绝对阈值（包含哪些关键词、参数、逻辑点）。
2. **定位证据**：在 {{RESPONSE}} 中搜寻支持或反驳该考点的具体文本片段。
3. **二元裁决**：
   - 若证据充分且完全匹配考点要求，判定为 is_met: true。
   - 若存在任何细微缺失、含糊或错误，判定为 is_met: false。
4. **计算分值**：严格按照 is_met 结果乘以 Weight。

# Input Slots (评估对象)

### [User Query]
{{QUERY}}

### [AI Response]
{{RESPONSE}}

### [Rubric List]
{{RUBRIC}}

# Output Format
请直接输出 JSON 对象，不要包含 Markdown 代码块标记（如 ```json ）。

```json
{
  "scoring_details": [
    {
      "rubric_index": 1,
      "description": "考点描述摘要",
      "weight": 5,
      "is_met": true,
      "score": 5,
     

In [ ]:
# result[i][generation_model][judge_model][j][k] : 第j次Judge，rubric[k]的打分
result = {}
for i,item in enumerate(first_100):
    result[i] = {} 
    query = item['prompt'][0]['content']

    
    # 把item['Rubrics']转换成字符串格式，方便后续替换到prompt中
    rubric_str = json.dumps(item['Rubrics'], ensure_ascii=False)
    print(rubric_str)

    for gen_mod in gen_models:
        response = responses[i][gen_mod]
        result[i][gen_mod] = {}
        
        for judge_mod in judge_models:
            result[i][gen_mod][judge_mod] = get_score(query,response,rubric_str,judge_mod,n=16)
    
    # break # for testing, 先judge一条就保存，后续再judge剩下的
            

[{"criterion": "The response contains no more than 50 words (including numbers and punctuation).", "points": 10}, {"criterion": "The summary clearly identifies NAME_1 and the pending decision regarding participation in the World Cross‑Country Championships.", "points": 10}, {"criterion": "The response is a concise summary that does not copy full sentences from the source text.", "points": 9}, {"criterion": "The summary explicitly states that the athlete is deciding whether to compete in the World Cross‑Country Championships because it may disrupt her London Marathon preparation.", "points": 9}, {"criterion": "The summary accurately states that NAME_1 has been granted extra time and that the decision is not yet final.", "points": 9}, {"criterion": "The summary mentions the deadline (Tuesday) or extra time granted for the decision.", "points": 8}, {"criterion": "The summary identifies the athlete as a former world cross‑country champion or indicates her high‑status experience.", "points"

In [12]:
# 保存result到result.json
json.dump(result, open("result.json", "w", encoding='utf-8'), ensure_ascii=False, indent=4)

### 3.2 Prompt改写对概率分布的影响
包括原始共l=5次，{0,response改写}*{0,Instruction改写}。
得到result[i][gen_mod][jud_mod][j][k][l]。